# Create Balzan Prize Awards (PRIZE PATTERN)

Balzan Prize winners from the official Balzan Foundation prizewinner archive and per-prizewinner detail pages.

**Prerequisite:** run `scripts/local/balzan_prize_to_s3.py` to fetch official archive/detail pages, verify row counts/collisions, and upload parquet to S3.

**Source:** `https://www.balzan.org/en/prizewinners` plus official five-year `quinquennio` archive pages and linked detail pages.

**S3 location:** `s3://openalex-ingest/awards/balzan_prize/balzan_prize_laureates.parquet`

**OpenAlex funder:** Fondazione Internazionale Premio Balzan, `funder_id = 4320310930`, ROR `https://ror.org/01xc6az48`, DOI `10.13039/100008995`.

**Schema notes:**
- Prize pattern: one row per official prizewinner record; shared award cards are split into one row per laureate.
- `lead_investigator` is the laureate.
- `funding_type = 'prize'`.
- Provenance is `balzan_prize`.
- Amount is NULL with currency `CHF` and an explicit prize-pattern waiver because the official archive/detail pages do not expose stable historical per-prizewinner amounts. Do not backfill historical amounts from third-party pages.
- `source_country` is preserved in the raw table but not mapped to `affiliation.country`, because it is the prizewinner nationality/country label rather than an institution country.
- Balzan does not publish laureate affiliations in a consistent structured field in the archive; `lead_investigator.affiliation` is NULL by design.

**Contractor handoff:** local validation was completed without AWS/S3 or Databricks credentials. Admin must upload the parquet, run this notebook, run QA, and then run `CreateAwards.ipynb`. No job YAML is included until admin verification.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.balzan_prize_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/balzan_prize/balzan_prize_laureates.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.balzan_prize_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.


In [ ]:
%sql
DESCRIBE openalex.awards.balzan_prize_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.balzan_prize_raw
LIMIT 10;


In [ ]:
%sql
-- Money-shaped columns from the raw source. Amount is intentionally NULL for
-- Balzan because historical per-prizewinner values are not exposed in the
-- official archive/detail pages.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'balzan_prize_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|fund|award|value|money|prize|chf';


In [ ]:
%sql
SELECT
    currency,
    amount_note,
    COUNT(*) AS rows,
    COUNT(source_award_amount) AS rows_with_amount
FROM openalex.awards.balzan_prize_raw
GROUP BY currency, amount_note
ORDER BY currency, amount_note;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(laureate_name) AS has_name,
    COUNT(award_year) AS has_year,
    COUNT(award_field) AS has_field,
    COUNT(citation) AS has_citation,
    COUNT(landing_page_url) AS has_landing_page_url,
    COUNT(research_project_url) AS has_research_project_url,
    COUNT(source_award_amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.balzan_prize_raw;


In [ ]:
%sql
SELECT
    award_year,
    award_field,
    laureate_name,
    source_country,
    winner_count,
    portion,
    citation,
    source_award_amount,
    currency,
    landing_page_url
FROM openalex.awards.balzan_prize_raw
ORDER BY TRY_CAST(award_year AS INT) DESC, award_field, laureate_name
LIMIT 50;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the `CROSS JOIN` would emit zero awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320310930;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.balzan_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320310930
),
normalized AS (
    SELECT
        n.*,
        TRY_CAST(n.award_year AS INT) AS award_year_int,
        TRY_CAST(n.source_award_amount AS DOUBLE) AS amount_double
    FROM openalex.awards.balzan_prize_raw n
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':balzan-prize:', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        CONCAT('Balzan Prize ', TRY_CAST(n.award_year_int AS STRING), ' - ', n.award_field, ' - ', n.laureate_name) AS display_name,
        NULLIF(n.citation, '') AS description,
        f.funder_id,
        n.funder_award_id,
        n.amount_double AS amount,
        NULLIF(n.currency, '') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        NULLIF(n.award_field, '') AS funder_scheme,
        'balzan_prize' AS provenance,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(TRY_CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            NULLIF(n.laureate_given_name, '') AS given_name,
            NULLIF(n.laureate_family_name, '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.laureate_name IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', TRY_CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


## Step 3: Insert Into `openalex_awards_raw` With Priority 76

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'balzan_prize' AND priority = 76;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    76 AS priority
FROM openalex.awards.balzan_prize_awards;


## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.balzan_prize_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.balzan_prize_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.balzan_prize_awards;


In [ ]:
%sql
-- Amount waiver: Balzan is a monetary prize, but the official archive/detail
-- pages do not expose stable historical per-prizewinner amount values.
-- amount should be NULL, currency should be CHF, and this waiver must be
-- reviewed during admin QA.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount_rows,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.balzan_prize_awards;


In [ ]:
%sql
SELECT
    funder.display_name AS funder_display_name,
    funder_id,
    COUNT(*) AS rows
FROM openalex.awards.balzan_prize_awards
GROUP BY funder.display_name, funder_id;


In [ ]:
%sql
SELECT
    funder_scheme,
    currency,
    COUNT(*) AS rows
FROM openalex.awards.balzan_prize_awards
GROUP BY funder_scheme, currency
ORDER BY funder_scheme, currency;


In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS rows
FROM openalex.awards.balzan_prize_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Duplicate checks. Both should return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.balzan_prize_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.balzan_prize_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT
    id,
    display_name,
    description,
    amount,
    currency,
    funder_scheme,
    start_year,
    lead_investigator,
    landing_page_url
FROM openalex.awards.balzan_prize_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 20;
